# ChronoMind ReVA Role 3 – full Transformer training

This reproducible workflow checks out the pinned QA and preprocessing revisions, generates the internal grouped split from official training annotations only, performs a short T4 runtime/memory preflight, and trains/evaluates the full Transformer model. The earlier 16/8 smoke run is complete; this workflow does not rerun it. Mean pooling remains gated until the Transformer results are reviewed.

## Reproducibility and run scope

Pinned revisions remain unchanged. The full run uses all 12,618 train and 3,155 validation QA rows, eight uniformly sampled frames, seed 17, Transformer temporal encoding, AdamW at 1e-4, five epochs, and batch size 4 for train and validation. Mixed precision is training-only; validation is FP32. Stop before training if the GPU is unavailable or the measured runtime estimate exceeds six hours. Path-only strict isolation does not establish source-sequence independence.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Check runtime settings.")

In [ ]:
import torch, platform
print(platform.python_version(), torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

In [ ]:
%pip -q install transformers huggingface_hub opencv-python-headless
# Colab provides PyTorch; no local model weights are bundled.

In [ ]:
import subprocess, os, json
from pathlib import Path
ROLE3_QA_REVISION = "74c417fb52fc21f923d788714875cbdb025d28aa"
SARTHAK_REVISION = "1a63b5009267a348b5fdd550b063869332d4ce0a"
DATASET_REVISION = "06224fa1b42e5e35677a8b7f2145e06307a9a4ee"
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
if ROLE3_QA_REVISION == "SET_AFTER_QA_COMMIT" or ROLE3_QA_REVISION == "62f383676b63717702ae27eb52e023a74c2e1d38": raise ValueError("Set ROLE3_QA_REVISION to the commit containing the QA modules; do not use the old Role 3 revision.")
REPO="https://github.com/sarthak-shastri/ChronoMind.git"
subprocess.run(["git","clone","--quiet",REPO,"/content/ChronoMind"],check=True)
subprocess.run(["git","-C","/content/ChronoMind","checkout","--detach",ROLE3_QA_REVISION],check=True)
subprocess.run(["git","clone","--quiet","--no-checkout",REPO,"/content/ChronoMind-sarthak"],check=True)
subprocess.run(["git","-C","/content/ChronoMind-sarthak","checkout","--detach",SARTHAK_REVISION],check=True)
os.environ['PYTHONPATH']="/content/ChronoMind:/content/ChronoMind-sarthak"
role3_head=subprocess.check_output(["git","-C","/content/ChronoMind","rev-parse","HEAD"],text=True).strip()
sarthak_head=subprocess.check_output(["git","-C","/content/ChronoMind-sarthak","rev-parse","HEAD"],text=True).strip()
if role3_head != ROLE3_QA_REVISION: raise RuntimeError(f"Role 3 checkout mismatch: expected {ROLE3_QA_REVISION}, got {role3_head}")
if sarthak_head != SARTHAK_REVISION: raise RuntimeError(f"Preprocessing checkout mismatch: expected {SARTHAK_REVISION}, got {sarthak_head}")
required_files=['scripts/train_reva_qa.py','scripts/evaluate_reva_qa.py','advanced_models/qa/data.py','advanced_models/qa/text_encoder.py','advanced_models/qa/multimodal_qa_model.py','advanced_models/temporal/temporal_encoder.py','advanced_models/vision/feature_extractor.py']
missing=[name for name in required_files if not (Path('/content/ChronoMind')/name).is_file()]
if missing: raise FileNotFoundError(f"Pinned QA revision is missing required files: {missing}")
print('Role3',role3_head)
print('Sarthak',sarthak_head)
print('Pinned historical Role 3 and CLIP revisions are used for reproducibility; branch HEAD and model main may advance.')

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
DATASET_REPO='ReVA-Benchmark/ReVA'
CACHE=Path('/content/reva_cache'); CACHE.mkdir(exist_ok=True)
official_train_json=hf_hub_download(DATASET_REPO,'train.json',repo_type='dataset',revision=DATASET_REVISION,cache_dir=CACHE)
train_data=json.loads(Path(official_train_json).read_text())
assert isinstance(train_data.get('QA'),list) and train_data['QA']
assert all(set(row.get('options',{}))=={'A','B','C','D'} for row in train_data['QA'][:5])
print('Pinned official train questions:',len(train_data['QA']))
print('Example ID/path:',train_data['QA'][0]['qa_id'],train_data['QA'][0]['video_path'])
# No official validation or test annotations are used to create the internal split.


In [ ]:
# Upload the full-training source overlay as one ZIP.
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, subprocess, sys, zipfile
uploaded=files.upload()
ZIP_NAME='role3_full_training_source.zip'
if ZIP_NAME not in uploaded:
    raise FileNotFoundError(f'Upload exactly {ZIP_NAME} from the project colab_package/ directory')
source_zip=Path('/content')/ZIP_NAME
source_zip.write_bytes(uploaded[ZIP_NAME])
SOURCE_ZIP_SHA256=hashlib.sha256(source_zip.read_bytes()).hexdigest()
package_root=Path('/content/ChronoMind')
with zipfile.ZipFile(source_zip) as zf:
    names=[item.filename for item in zf.infolist() if not item.is_dir()]
    if len(names)!=len(set(names)) or 'manifest.json' not in names:
        raise ValueError('Source ZIP has duplicate paths or no manifest.json')
    manifest_bytes=zf.read('manifest.json')
    PACKAGE_MANIFEST_SHA256=hashlib.sha256(manifest_bytes).hexdigest()
    manifest=json.loads(manifest_bytes)
    if manifest['base_qa_commit']!=ROLE3_QA_REVISION or manifest['preprocessing_commit']!=SARTHAK_REVISION:
        raise ValueError('Source manifest revisions do not match the pinned notebook revisions')
    if manifest['dataset']['revision']!=DATASET_REVISION or manifest['clip']['revision']!=CLIP_REVISION:
        raise ValueError('Source manifest dataset/CLIP revision mismatch')
    expected={item['path']:item for item in manifest['files']}
    if set(names)!=(set(expected)|{'manifest.json'}):
        raise ValueError('ZIP members differ from the manifest file list')
    OVERLAY_SHA256={}
    for rel,item in expected.items():
        posix=PurePosixPath(rel)
        if posix.is_absolute() or '..' in posix.parts:
            raise ValueError(f'Unsafe archive path: {rel}')
        target=(package_root/Path(*posix.parts)).resolve()
        if package_root.resolve() not in target.parents:
            raise ValueError(f'Archive path escapes repository: {rel}')
        payload=zf.read(rel)
        digest=hashlib.sha256(payload).hexdigest()
        if digest!=item['sha256']:
            raise ValueError(f'ZIP payload hash mismatch: {rel}')
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(payload)
        if hashlib.sha256(target.read_bytes()).hexdigest()!=item['sha256']:
            raise ValueError(f'Extracted file hash mismatch: {rel}')
        OVERLAY_SHA256[rel]=digest
Path('/content/role3_experiment_manifest.json').write_bytes(manifest_bytes)
train_help=subprocess.check_output([sys.executable,str(package_root/'scripts/train_reva_qa.py'),'--help'],text=True)
eval_help=subprocess.check_output([sys.executable,str(package_root/'scripts/evaluate_reva_qa.py'),'--help'],text=True)
assert '--temporal-encoder' in train_help and 'path_only_strict' in train_help
assert 'path_only_strict' in eval_help
print('Verified full-training overlay files:',len(OVERLAY_SHA256))
print('Source ZIP SHA-256:',SOURCE_ZIP_SHA256)
print('Manifest SHA-256:',PACKAGE_MANIFEST_SHA256)
print('Temporal encoder and path_only_strict CLI options: available')

In [ ]:
# Use only the pinned official train.json to create the deterministic group split.
from pathlib import Path
import os, subprocess, sys
INTERNAL_SPLIT_DIR=Path('/content/reva_internal_80_20')
split_script=Path('/content/ChronoMind/scripts/create_reva_internal_grouped_split.py')
split_cmd=[sys.executable,str(split_script),'--train',official_train_json,'--output-dir',str(INTERNAL_SPLIT_DIR),'--seed','42']
VIDEO_ROOT=os.environ.get('REVA_VIDEO_ROOT')
if VIDEO_ROOT: split_cmd.extend(['--video-root',VIDEO_ROOT])
subprocess.run(split_cmd,check=True)
split_stats=json.loads((INTERNAL_SPLIT_DIR/'split_statistics.json').read_text())
assert split_stats['normalized_video_path_overlap_count']==0
print('Actual split counts:',split_stats['train_questions'],'train /',split_stats['validation_questions'],'validation')
print('Unique paths:',split_stats['train_unique_video_paths'],'train /',split_stats['validation_unique_video_paths'],'validation')
print('Normalized path overlap:',split_stats['normalized_video_path_overlap_count'])
if split_stats['available_video_hashing']:
    print('Hash coverage:',split_stats['hashed_unique_video_paths'],'unique paths; missing files:',split_stats['missing_video_file_count'])
    print('Identical-file groups:',len(split_stats['identical_file_hash_groups']),'cross-split hash overlap:',len(split_stats['cross_split_identical_hash_overlap']))
else:
    print('Video hashing/missing-file counts: NOT CHECKED (REVA_VIDEO_ROOT is unset).')
print('Source-sequence limitation:',split_stats['source_sequence_independence'])


In [ ]:
# Apply the generated internal annotation files and strictly preflight path isolation.
from pathlib import Path
import sys
sys.path.insert(0,'/content/ChronoMind')
from scripts.reva_split_preflight import preflight_splits
internal_train=INTERNAL_SPLIT_DIR/'train_internal.json'
internal_val=INTERNAL_SPLIT_DIR/'val_internal.json'
assert internal_train.is_file() and internal_val.is_file()
train_json=str(internal_train); val_json=str(internal_val)
train_data=json.loads(internal_train.read_text()); val_data=json.loads(internal_val.read_text())
assert len(train_data['QA'])==split_stats['train_questions']
assert len(val_data['QA'])==split_stats['validation_questions']
path_audit=preflight_splits(train_json,val_json,video_root=VIDEO_ROOT)
if path_audit['path_only_issues']:
    raise RuntimeError('Path-only strict preflight failed: '+ '; '.join(path_audit['path_only_issues'][:8]))
print('Path-only strict preflight passed; source-sequence isolation is not asserted.')
if path_audit['unverifiable_source_sequence_relationships']['warning']:
    print('WARNING:',path_audit['unverifiable_source_sequence_relationships']['warning'])
print('Train/validation records:',len(train_data['QA']),len(val_data['QA']))


In [ ]:
import sys
sys.path.insert(0,'/content/ChronoMind'); sys.path.insert(0,'/content/ChronoMind-sarthak')
from preprocessing import ReVADataset
from advanced_models.qa.text_encoder import CLIPTextFeatureExtractor
from advanced_models.qa.multimodal_qa_model import MultimodalVideoQAModel
from advanced_models.temporal.temporal_encoder import TransformerTemporalEncoder
from advanced_models.vision.feature_extractor import CLIPVisionFeatureExtractor
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vision=CLIPVisionFeatureExtractor.from_pretrained(input_normalization='imagenet',frame_chunk_size=4,cuda_mixed_precision=device.type=='cuda',revision=CLIP_REVISION)
text=CLIPTextFeatureExtractor.from_pretrained(revision=CLIP_REVISION)
model=MultimodalVideoQAModel(vision,text,TransformerTemporalEncoder(512,num_heads=8,num_layers=2,max_seq_len=16)).to(device)
print('Frozen encoders loaded:',device)

In [ ]:
from preprocessing import ReVADataset
from advanced_models.qa.data import ReVAQADatasetAdapter,collate_qa
smoke=ReVAQADatasetAdapter(ReVADataset(val_json,num_frames=4,split=None,sample_ids=[val_data['QA'][0]['qa_id']],dataset_repo_id=DATASET_REPO,revision=DATASET_REVISION,missing_video='raise'))
batch=collate_qa([smoke[0]])
canonical_answer=val_data['QA'][0]['correct_answer']
assert batch['answers']==[canonical_answer]
assert batch['labels'].tolist()==['ABCD'.index(canonical_answer)]
with torch.inference_mode(): print('forward logits',model(batch['video_frames'].to(device),batch['questions'],batch['choices'])['answer_logits'].shape)

## Package and integrity behavior

Upload the single `role3_full_training_source.zip` from `colab_package/` when prompted. The notebook extracts only manifest-listed repository paths, verifies every file SHA-256 and all pinned revisions, then checks that `--temporal-encoder` and `path_only_strict` are available. The strict mode rejects normalized video-path overlap, invalid/missing train labels/options, duplicate IDs, missing paths, and available cross-split file-hash overlap. It warns that path isolation does not establish source-sequence independence.

The notebook downloads only the pinned official `train.json`; it derives the internal validation split from those train rows. Hash coverage and missing-video counts are printed as unavailable unless `REVA_VIDEO_ROOT` points to a local video snapshot. The opt-in full mean-pooling/Transformer comparison uses separate output directories and writes a comparison JSON only after both runs and evaluations complete.


## Full Transformer run and gated comparison

The benchmark cell runs a short real-data throughput and GPU-memory preflight on representative training batches. It estimates the full five-epoch run; the next cell refuses to train when CUDA is unavailable or the estimate exceeds six hours. Training and evaluation use the complete generated split, with no sample limit. The evaluator writes full per-question predictions and aggregate metrics into `/content/role3_full_transformer/`.

Frame embeddings are cached in process-local CPU FP32 tensors, keyed by video path plus a fingerprint of the pinned dataset, preprocessing, CLIP, frame-sampling, geometry, and normalization settings. Text embeddings are cached by exact input string in a process-local bounded LRU. Neither cache survives the process; recreate them after changing any keyed revision or preprocessing option. The cache does not store labels or feed answers to the model.

The mean-pooling cell is disabled until `TRANSFORMER_RESULTS_REVIEWED=True` is set after reviewing the full Transformer results. It uses the same splits, seed, frame count, batch sizes, epochs, learning rate, and evaluation procedure.

In [ ]:
# Runtime and memory preflight with cold-vs-warm cache profiling; no dataset is truncated.
import gc, math, time, statistics
from pathlib import Path
import numpy as np
from torch.utils.data import DataLoader
from advanced_models.qa.data import ReVAQADatasetAdapter, collate_qa
from preprocessing import ReVADataset

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required; full training will not run on CPU.')

gpu_name = torch.cuda.get_device_name(0)
gpu_free_bytes, gpu_total_bytes = torch.cuda.mem_get_info()
print(f'GPU: {gpu_name} | Free: {gpu_free_bytes/2**30:.2f} GiB / Total: {gpu_total_bytes/2**30:.2f} GiB')

# Configure local disk frame cache to eliminate repeated OpenCV decodes across 5 epochs
FRAME_CACHE_DIR = Path('/content/reva_frame_cache')
FRAME_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PREFLIGHT_BATCH_SIZE = 8
PREFLIGHT_COUNT = min(64, len(train_data['QA']))
preflight_ids = [str(row['qa_id']) for row in train_data['QA'][:PREFLIGHT_COUNT]]
preflight_ds = ReVAQADatasetAdapter(ReVADataset(
    train_json, num_frames=8, sampling='uniform', image_size=(224, 224),
    normalize=True, split=None, cache_dir=str(FRAME_CACHE_DIR), sample_ids=preflight_ids,
    dataset_repo_id=DATASET_REPO, revision=DATASET_REVISION, missing_video='raise'
))
preflight_loader = DataLoader(
    preflight_ds, batch_size=PREFLIGHT_BATCH_SIZE, shuffle=False,
    collate_fn=collate_qa, num_workers=2, pin_memory=True, persistent_workers=True
)
preflight_opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

model.train()
model.vision.cuda_mixed_precision = True
torch.cuda.reset_peak_memory_stats()

# Pass 1: Cold cache profiling (first-time video decoding + CLIP encoding on unseen videos)
cold_times = []
print('--- PASS 1: Cold Cache Profiling (Initial Decode & Embed) ---')
for step, b in enumerate(preflight_loader):
    start = time.perf_counter()
    preflight_opt.zero_grad(set_to_none=True)
    with torch.autocast('cuda', dtype=torch.float16):
        result = model(b['video_frames'].to(device, non_blocking=True), b['questions'], b['choices'], video_cache_keys=b['video_paths'])
        loss = torch.nn.functional.cross_entropy(result['answer_logits'], b['labels'].to(device))
    loss.backward()
    preflight_opt.step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    cold_times.append(elapsed)
    print(f'Cold batch {step+1}: {elapsed:.3f}s | loss: {loss.item():.4f}')

# Pass 2: Warm cache profiling (FrameCache hits from disk + CLIP embedding cache hits)
warm_times = []
print('--- PASS 2: Warm Cache Profiling (FrameCache + CLIP Embedding Hits) ---')
for step, b in enumerate(preflight_loader):
    start = time.perf_counter()
    preflight_opt.zero_grad(set_to_none=True)
    with torch.autocast('cuda', dtype=torch.float16):
        result = model(b['video_frames'].to(device, non_blocking=True), b['questions'], b['choices'], video_cache_keys=b['video_paths'])
        loss = torch.nn.functional.cross_entropy(result['answer_logits'], b['labels'].to(device))
    loss.backward()
    preflight_opt.step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    warm_times.append(elapsed)
    print(f'Warm batch {step+1}: {elapsed:.3f}s | loss: {loss.item():.4f}')

peak_bytes = torch.cuda.max_memory_allocated()
print(f'\nPeak allocated GPU memory: {peak_bytes/2**30:.2f} GiB (Headroom: {(gpu_total_bytes - peak_bytes)/2**30:.2f} GiB)')

# Statistics
mean_cold = statistics.mean(cold_times)
median_cold = statistics.median(cold_times)
mean_warm = statistics.mean(warm_times)
median_warm = statistics.median(warm_times)
p95_warm = float(np.percentile(warm_times, 95))

print(f'Cold batch stats: mean={mean_cold:.3f}s, median={median_cold:.3f}s')
print(f'Warm batch stats: mean={mean_warm:.3f}s, median={median_warm:.3f}s, p95={p95_warm:.3f}s')
print(f'Visual cache stats: {model.vision.feature_cache_stats()}')
print(f'Text cache stats: {model.text.feature_cache_stats()}')

# Accurate projection based on unique video counts and sample retrievals
unique_train_vids = len(set(row['video_path'] for row in train_data['QA']))
unique_val_vids = len(set(row['video_path'] for row in val_data['QA']))
total_unique_vids = unique_train_vids + unique_val_vids

train_steps = math.ceil(len(train_data['QA']) / PREFLIGHT_BATCH_SIZE)
val_steps = math.ceil(len(val_data['QA']) / PREFLIGHT_BATCH_SIZE)
PROJECTED_EPOCHS = 5

# Cold cost (OpenCV decode + CLIP embed) occurs ONCE per unique video (1,045 total)
cold_cost_per_vid = max(0.0, (mean_cold - mean_warm) / PREFLIGHT_BATCH_SIZE)
total_cold_seconds = total_unique_vids * cold_cost_per_vid

# Warm cost applies to all training and validation steps across all 5 epochs
total_train_warm_seconds = train_steps * PROJECTED_EPOCHS * mean_warm
total_val_warm_seconds = val_steps * PROJECTED_EPOCHS * (mean_warm * 0.65)
total_seconds = total_cold_seconds + total_train_warm_seconds + total_val_warm_seconds
projected_hours = total_seconds / 3600

print(f'\n=== ACCURATE FULL-RUN RUNTIME PROJECTION (5 EPOCHS, BATCH SIZE {PREFLIGHT_BATCH_SIZE}) ===')
print(f'Unique videos: {total_unique_vids} ({unique_train_vids} train + {unique_val_vids} val)')
print(f'One-time cold video extraction total: {total_cold_seconds/60:.1f} minutes')
print(f'Warm training total ({PROJECTED_EPOCHS} epochs, {train_steps*PROJECTED_EPOCHS} steps): {total_train_warm_seconds/60:.1f} minutes')
print(f'Warm validation total ({PROJECTED_EPOCHS} epochs, {val_steps*PROJECTED_EPOCHS} steps): {total_val_warm_seconds/60:.1f} minutes')
print(f'Projected Total Runtime: {projected_hours:.2f} hours ({total_seconds/60:.1f} minutes)')

if projected_hours > 6.0:
    raise RuntimeError(f'Projected {projected_hours:.2f} hours exceeds 6-hour safety budget; stopping.')
print('Preflight passed within the 6-hour safety budget! Model released for full training.')

del preflight_opt, preflight_loader, preflight_ds, b, result, loss, model, vision, text
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Full 12,618 / 3,155 Transformer training with batch_size=8, frame caching and resumable checkpoints.
import math, subprocess
from pathlib import Path

FULL_OUTPUT = Path('/content/role3_full_transformer')
FULL_OUTPUT.mkdir(parents=True, exist_ok=True)
FRAME_CACHE_DIR = Path('/content/reva_frame_cache')
FRAME_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FULL_EPOCHS = 5
FULL_LR = 1e-4
FULL_BATCH_SIZE = 8
FULL_VAL_BATCH_SIZE = 8

train_cmd = [
    sys.executable, '/content/ChronoMind/scripts/train_reva_qa.py',
    '--train-annotations', train_json,
    '--val-annotations', val_json,
    '--dataset-repo-id', DATASET_REPO,
    '--revision', DATASET_REVISION,
    '--preprocessing-root', '/content/ChronoMind-sarthak',
    '--frames', '8',
    '--batch-size', str(FULL_BATCH_SIZE),
    '--val-batch-size', str(FULL_VAL_BATCH_SIZE),
    '--num-workers', '2',
    '--temporal-encoder', 'transformer',
    '--split-integrity-mode', 'path_only_strict',
    '--epochs', str(FULL_EPOCHS),
    '--lr', str(FULL_LR),
    '--seed', '17',
    '--mixed-precision',
    '--clip-revision', CLIP_REVISION,
    '--frame-cache-dir', str(FRAME_CACHE_DIR),
    '--output', str(FULL_OUTPUT)
]

# Support automatic resume if previous checkpoint exists
latest_ckpt = FULL_OUTPUT / 'latest.pt'
if latest_ckpt.is_file():
    print(f'Found existing checkpoint {latest_ckpt}; resuming training.')
    train_cmd.extend(['--resume', str(latest_ckpt)])

if VIDEO_ROOT:
    train_cmd.extend(['--video-root', VIDEO_ROOT])

print('Starting training command:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)
print('Full training completed:', FULL_OUTPUT)


In [ ]:
# Evaluate all validation rows with FP32 inference and save into the full experiment directory.
FRAME_CACHE_DIR = Path('/content/reva_frame_cache')
eval_cmd = [
    sys.executable, '/content/ChronoMind/scripts/evaluate_reva_qa.py',
    '--annotations', val_json,
    '--train-annotations', train_json,
    '--checkpoint', str(FULL_OUTPUT / 'best.pt'),
    '--preprocessing-root', '/content/ChronoMind-sarthak',
    '--dataset-repo-id', DATASET_REPO,
    '--revision', DATASET_REVISION,
    '--batch-size', str(FULL_VAL_BATCH_SIZE),
    '--split-integrity-mode', 'path_only_strict',
    '--frame-cache-dir', str(FRAME_CACHE_DIR),
    '--output-dir', str(FULL_OUTPUT)
]
if VIDEO_ROOT: eval_cmd.extend(['--video-root', VIDEO_ROOT])
subprocess.run(eval_cmd, check=True)
print('Full validation predictions and metrics saved:', FULL_OUTPUT)


In [ ]:
# Verify complete validation coverage, labels, checkpoint hash, metrics, and summarize errors.
import hashlib, json, torch
from collections import Counter
from pathlib import Path
history=json.loads((FULL_OUTPUT/'history.json').read_text())
metrics=json.loads((FULL_OUTPUT/'metrics.json').read_text())
pred_payload=json.loads((FULL_OUTPUT/'predictions.json').read_text())
metadata=json.loads((FULL_OUTPUT/'experiment_metadata.json').read_text())
expected_rows=val_data['QA']; expected_ids=[str(row['qa_id']) for row in expected_rows]
predictions=pred_payload['predictions']; pred_ids=[row['qa_id'] for row in predictions]
assert len(expected_ids)==len(set(expected_ids))==3155
assert len(pred_ids)==len(set(pred_ids))==len(expected_ids)
assert pred_ids==expected_ids
assert metrics['sample_count']==metrics['labeled_count']==len(expected_ids)
assert all(row['ground_truth']==src['correct_answer'] for row,src in zip(predictions,expected_rows))
assert all(row['options']=={k:src['options'][k] for k in 'ABCD'} for row,src in zip(predictions,expected_rows))
assert all(row['predicted_answer'] in 'ABCD' and row['correctness'] in ('correct','incorrect') for row in predictions)
assert sum(metrics['prediction_distribution'].values())==len(predictions)
assert sum(v['total'] for v in metrics['category_accuracy'].values())==len(expected_ids)
assert sum(v['total'] for v in metrics['task_accuracy'].values())==len(expected_ids)
checkpoint_path=FULL_OUTPUT/'best.pt'
checkpoint_hash=hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
assert checkpoint_hash==metadata['checkpoint_sha256']==metrics['checkpoint_sha256']==pred_payload['model']['metadata']['checkpoint_sha256']
checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
assert checkpoint['model_config']['temporal_encoder_type']=='transformer'
best_row=max(history,key=lambda row:row['val_accuracy'])
assert abs(checkpoint['validation_accuracy']-best_row['val_accuracy'])<1e-8
assert abs(metrics['accuracy']-checkpoint['validation_accuracy'])<1e-8
required=['best.pt','history.json','predictions.json','metrics.json','experiment_metadata.json']
assert all((FULL_OUTPUT/name).is_file() for name in required)
wrong=[row for row in predictions if row['correctness']=='incorrect']
print('Coverage:',len(predictions),'/',len(expected_ids),'validation QA IDs; checkpoint SHA-256:',checkpoint_hash)
print('Best epoch',checkpoint['epoch'],'train loss/accuracy',best_row['train_loss'],best_row['train_accuracy'],'validation loss/accuracy',best_row['val_loss'],best_row['val_accuracy'])
print('Best-checkpoint validation accuracy:',metrics['accuracy'],'category accuracy:',json.dumps(metrics['category_accuracy']))
print('Task accuracy:',json.dumps(metrics['task_accuracy']))
print('Confusion matrix A/B/C/D:',metrics['confusion_matrix'],'prediction distribution:',metrics['prediction_distribution'])
print('Mean/median inference sec per sample:',metrics['mean_inference_time_seconds'],metrics['median_inference_time_seconds'])
print('Peak CUDA memory allocated bytes:',metadata.get('peak_cuda_memory_bytes'))
print('Incorrect examples:',json.dumps([{'qa_id':r['qa_id'],'category':r['category'],'task':r['task'],'ground_truth':r['ground_truth'],'predicted':r['predicted_answer'],'question':r['question']} for r in wrong[:20]],ensure_ascii=False))
print('All experiment artifacts verified in',FULL_OUTPUT)

In [ ]:
# Mean-pooling follow-up is deliberately gated until the full Transformer results are reviewed.
TRANSFORMER_RESULTS_REVIEWED=False
if TRANSFORMER_RESULTS_REVIEWED:
    import json, subprocess, sys
    from pathlib import Path
    mean_output=Path('/content/role3_full_mean_pooling')
    if mean_output.exists() and any(mean_output.iterdir()):
        raise FileExistsError(f'{mean_output} already contains files; refusing to overwrite.')
    mean_cmd=[sys.executable,'/content/ChronoMind/scripts/train_reva_qa.py',
        '--train-annotations',train_json,'--val-annotations',val_json,
        '--dataset-repo-id',DATASET_REPO,'--revision',DATASET_REVISION,
        '--preprocessing-root','/content/ChronoMind-sarthak','--frames','8',
        '--batch-size',str(FULL_BATCH_SIZE),'--val-batch-size',str(FULL_VAL_BATCH_SIZE),
        '--num-workers','2','--temporal-encoder','mean_pooling','--split-integrity-mode','path_only_strict',
        '--epochs',str(FULL_EPOCHS),'--lr',str(FULL_LR),'--seed','17','--mixed-precision',
        '--clip-revision',CLIP_REVISION,'--output',str(mean_output)]
    if VIDEO_ROOT: mean_cmd.extend(['--video-root',VIDEO_ROOT])
    subprocess.run(mean_cmd,check=True)
    mean_eval=[sys.executable,'/content/ChronoMind/scripts/evaluate_reva_qa.py',
        '--annotations',val_json,'--train-annotations',train_json,'--checkpoint',str(mean_output/'best.pt'),
        '--preprocessing-root','/content/ChronoMind-sarthak','--dataset-repo-id',DATASET_REPO,
        '--revision',DATASET_REVISION,'--batch-size',str(FULL_VAL_BATCH_SIZE),
        '--split-integrity-mode','path_only_strict','--output-dir',str(mean_output)]
    if VIDEO_ROOT: mean_eval.extend(['--video-root',VIDEO_ROOT])
    subprocess.run(mean_eval,check=True)
    transformer_metrics=json.loads((FULL_OUTPUT/'metrics.json').read_text())
    mean_metrics=json.loads((mean_output/'metrics.json').read_text())
    assert mean_metrics['sample_count']==len(val_data['QA'])
    comparison={'dataset_revision':DATASET_REVISION,'preprocessing_revision':SARTHAK_REVISION,
        'clip_revision':CLIP_REVISION,'split_seed':42,'training_seed':17,'frames':8,
        'train_batch_size':FULL_BATCH_SIZE,'validation_batch_size':FULL_VAL_BATCH_SIZE,
        'epochs':FULL_EPOCHS,'learning_rate':FULL_LR,
        'transformer_accuracy':transformer_metrics['accuracy'],'mean_pooling_accuracy':mean_metrics['accuracy'],
        'transformer_checkpoint_sha256':transformer_metrics['checkpoint_sha256'],
        'mean_pooling_checkpoint_sha256':mean_metrics['checkpoint_sha256'],
        'note':'Internal validation comparison only; no official test performance claim.'}
    Path('/content/role3_encoder_comparison.json').write_text(json.dumps(comparison,indent=2))
    print('Completed fair mean-pooling comparison:',json.dumps(comparison))
else:
    print('Mean-pooling run not started. Review the full Transformer results, then set TRANSFORMER_RESULTS_REVIEWED=True.')

In [ ]:
from pathlib import Path
output_dir=Path('/content/role3_full_transformer')
print('Output directory exists:',output_dir.exists())
if output_dir.exists():
    for file in sorted(output_dir.rglob('*')):
        if file.is_file(): print(file.relative_to(output_dir),f'{file.stat().st_size/1024:.2f} KB')

In [ ]:
# Package all full Transformer artifacts into a single archive for download
import zipfile
from pathlib import Path
from google.colab import files

output_dir = Path('/content/role3_full_transformer')
archive_zip = Path('/content/role3_full_transformer_artifacts.zip')
if archive_zip.exists():
    archive_zip.unlink()

with zipfile.ZipFile(archive_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(output_dir.rglob('*')):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(output_dir))

print('Created', archive_zip, 'size bytes:', archive_zip.stat().st_size)
files.download(str(archive_zip))